# 06 — Segmentación con SAM

## ¿Qué vamos a construir hoy?

Generarás máscaras de segmentación precisas combinando YOLO (detección)
con SAM 3 (segmentación), y verás que Supervision funciona igual
sin importar qué modelo uses.

**Aprenderás a:**
- Entender la diferencia entre bounding box y máscara de segmentación
- Usar SAM 3 con prompts de bounding box para segmentar objetos
- Usar prompts de texto de SAM 3 para segmentar sin detección previa
- Ver el agnosticismo de framework de Supervision en acción

**Tiempo estimado:** 35 minutos

## ⏱️ Estructura de la Clase (Duración estimada: 1 hora)
- **Introducción y Conceptos Base**: 15 min
- **Desarrollo y Demostración Práctica**: 25 min
- **Análisis y Casos Extremos (Pausa y Observa)**: 20 min

## Bounding box vs. Máscara de segmentación

**Bounding box:** un rectángulo que enmarca al objeto.
Rápido de calcular, pero incluye píxeles del fondo dentro del rectángulo.

**Máscara de segmentación:** una imagen booleana del tamaño de la imagen original.
`True` en cada píxel que pertenece al objeto, `False` en el fondo.
Resultado: la silueta exacta del objeto.

Si bounding box es recortar con tijeras rectas,
SAM es recortar siguiendo exactamente el contorno del objeto.

```
Bounding box:   ████████
                █ perro █   ← incluye fondo
                ████████

Máscara:          ██
                ██████       ← solo el objeto
                  ████
```

## La máscara como estructura de datos

Una máscara de segmentación es una **matriz NumPy booleana** del mismo tamaño
que la imagen original: un valor `True` o `False` por cada píxel.

```python
# Para una imagen de 480 × 640 px con 3 objetos detectados:
detections.mask.shape  # → (3, 480, 640)
                       #      ↑    ↑     ↑
                       #   N obj  alto  ancho
```

- `True` (o `1`) → ese píxel **pertenece al objeto**.
- `False` (o `0`) → ese píxel **es fondo**.

Operaciones directas con NumPy:
```python
mascara = detections.mask[0]       # máscara del primer objeto (shape: alto × ancho)
area    = mascara.sum()            # número de píxeles del objeto
fraccion = area / mascara.size     # qué fracción de la imagen ocupa el objeto
recorte = imagen.copy()
recorte[~mascara] = 0              # poner a negro todo lo que NO es el objeto
```

La precisión es a nivel de **píxel individual** — muy diferente al rectángulo del
bounding box que incluye fondo. Esto es clave para medir área real, eliminar el
fondo o generar datasets de entrenamiento con siluetas exactas.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install supervision ultralytics
import supervision as sv
from ultralytics import YOLO, SAM
import cv2
import numpy as np
import matplotlib.pyplot as plt
import urllib.request
from pathlib import Path

Path("assets").mkdir(exist_ok=True)
urllib.request.urlretrieve("https://ultralytics.com/images/bus.jpg",    "assets/bus.jpg")
urllib.request.urlretrieve("https://ultralytics.com/images/zidane.jpg", "assets/zidane.jpg")

image = cv2.imread("assets/bus.jpg")
print(f"Imagen cargada: {image.shape}")

## Paso 1: Detectar objetos con YOLO

Las coordenadas de YOLO son las "pistas" que le damos a SAM
para que sepa dónde mirar en la imagen.

In [ ]:
yolo_model = YOLO("yolov8n.pt")
yolo_results = yolo_model(image)[0]
yolo_detections = sv.Detections.from_ultralytics(yolo_results)

print(f"YOLO detectó {len(yolo_detections)} objetos")

# Visualizar bounding boxes antes de segmentar
box_annotator = sv.BoxAnnotator()
scene_yolo = box_annotator.annotate(scene=image.copy(), detections=yolo_detections)
plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(scene_yolo, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("Paso 1: Bounding boxes de YOLO — las 'pistas' para SAM")
plt.show()

## Paso 2: Generar máscaras con SAM 3

SAM 3 usa las cajas de YOLO como prompts: le dicen "busca un objeto aquí"
y SAM genera la silueta exacta de ese objeto.

SAM 3 también acepta prompts de texto — lo veremos en el Experimento 4.


In [ ]:
sam_path = "/content/drive/MyDrive/RandD/Archive_Zero_Resolved/sam3.pt"
sam_model = SAM(sam_path)
# sam3.pt ≈ 3.4 GB — incluido en ultralytics, no requiere paquete extra
# Si no se descarga automáticamente:
#   1. Solicita acceso en https://huggingface.co/facebook/sam3
#   2. Descarga sam3.pt y colócalo en la carpeta del notebook

# SAM espera una lista de Python, no un array NumPy
# .tolist() hace la conversión
bboxes = yolo_detections.xyxy.tolist()

sam_results = sam_model(image, bboxes=bboxes)[0]

# La misma función from_ultralytics() funciona para YOLO y para SAM
# — ese es el punto central de Supervision: no importa qué modelo usaste
sam_detections = sv.Detections.from_ultralytics(sam_results)

print(f"Detecciones SAM: {len(sam_detections)}")
print(f"¿Tiene máscaras? {sam_detections.mask is not None}")
if sam_detections.mask is not None:
    print(f"Shape de las máscaras: {sam_detections.mask.shape}")
    # (N_objetos, alto_imagen, ancho_imagen)
    # Cada "capa" es una máscara booleana para un objeto


## Paso 3: Visualización Raw de Máscaras

En lugar de usar un visualizador avanzado aquí, vamos a dibujar la máscara directamente usando `matplotlib` para entenderla matemáticamente.
**Nota:** El uso avanzado de `sv.MaskAnnotator` (con opacidades, colores por clase, etc.) lo veremos a fondo en la clase **03_b_f**.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

if 'sam_detections' in locals() and sam_detections.mask is not None:
    primera_mascara = sam_detections.mask[0]
    plt.imshow(primera_mascara, cmap='gray')
    plt.title('Máscara en Bruto (Matplotlib)')
    plt.axis('off')
    plt.show()

## Pausa y observa: ¿Qué hay dentro de una máscara?

In [ ]:
if sam_detections.mask is not None:
    primera_mascara = sam_detections.mask[0]  # máscara del primer objeto
    
    print(f"Tipo: {type(primera_mascara)}")
    print(f"Shape: {primera_mascara.shape}")     # (alto, ancho) — del tamaño de la imagen
    print(f"Valores: {np.unique(primera_mascara)}")  # solo True y False
    print(f"Píxeles del objeto: {primera_mascara.sum()}")
    print(f"Píxeles totales:    {primera_mascara.size}")
    
    # Recortar solo el objeto usando la máscara como "molde"
    objeto_recortado = image.copy()
    # ~ invierte la máscara: ponemos en negro todos los píxeles que NO son el objeto
    objeto_recortado[~primera_mascara] = 0
    
    plt.figure(figsize=(8, 6))
    plt.imshow(cv2.cvtColor(objeto_recortado, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.title("Objeto recortado con precisión de píxel")
    plt.show()

### Experimento 4: Texto como prompt — adelanto

SAM 3 también acepta prompts de texto: puedes pedirle que segmente
"todas las personas" sin necesitar un detector previo.

> **Esto lo veremos en profundidad en NB07.** Aquí solo un adelanto de cómo se ve la llamada.


In [ ]:
# Adelanto — el código completo está en NB07
# from ultralytics.models.sam import SAM3SemanticPredictor
# predictor = SAM3SemanticPredictor(overrides=dict(conf=0.25, task="segment", mode="predict", model="sam3.pt"))
# predictor.set_image(image)
# resultados_texto = predictor(text=["person"])[0]
# det_texto = sv.Detections.from_ultralytics(resultados_texto)
#
# → En NB07 explorarás esto con múltiples conceptos, umbrales y la comparación texto vs. bbox.


## 🚀 Reto de extensión

**Tarea:** Calcula para cada objeto el porcentaje del bounding box que realmente pertenece al objeto (área de la máscara / área del bounding box).

Un porcentaje alto indica que la caja está bien ajustada al objeto.
Un porcentaje bajo indica que la caja incluye mucho fondo.

**Pista:**
```python
if sam_detections.mask is not None:
    for i in range(len(sam_detections)):
        area_mascara = sam_detections.mask[i].sum()
        area_caja    = sam_detections.box_area[i]
        porcentaje   = area_mascara / area_caja * 100
        clase        = yolo_results.names[sam_detections.class_id[i]] if sam_detections.class_id is not None else "?"
        print(f"Objeto {i} ({clase}): {porcentaje:.1f}% del bounding box es el objeto")
```

In [ ]:
# Escribe tu solución aquí
if sam_detections.mask is not None:
    for i in range(len(sam_detections)):
        area_mascara = sam_detections.mask[i].sum()          # píxeles True en la máscara
        area_caja    = sam_detections.box_area[i]            # (x2-x1) × (y2-y1) en píxeles
        porcentaje   = area_mascara / area_caja * 100        # completa esta línea
        etiqueta     = f"obj {i}: {porcentaje:.1f}%"         # puedes cambiar el formato
        # print(etiqueta)

## 💾 Guardar máscaras en JSON

Las máscaras de segmentación son arrays NumPy booleanos. Para guardarlas en JSON
se codifican en base64 — formato compacto y portable que puede leerse desde
cualquier lenguaje de programación.

In [ ]:
import json, base64
import numpy as np

def detections_to_dict_with_masks(detections, class_names=None):
    """Convierte sv.Detections (con máscaras) a un dict JSON-compatible."""
    data = {
        "xyxy":        detections.xyxy.tolist(),
        "confidence":  detections.confidence.tolist() if detections.confidence is not None else None,
        "class_id":    detections.class_id.tolist()   if detections.class_id   is not None else None,
        "class_names": [class_names[c] for c in detections.class_id]
                       if (class_names and detections.class_id is not None) else None,
    }
    if detections.mask is not None:
        H, W = detections.mask.shape[1:]
        data["mask_shape"] = [H, W]
        # Cada máscara se comprime con packbits y se codifica en base64
        data["masks_b64"] = [
            base64.b64encode(np.packbits(m.flatten()).tobytes()).decode()
            for m in detections.mask
        ]
    return data

resultado = detections_to_dict_with_masks(sam_detections, class_names=yolo_results.names)

with open("assets/predicciones_mascaras.json", "w", encoding="utf-8") as f:
    json.dump(resultado, f, indent=2, ensure_ascii=False)

print(f"Guardado: assets/predicciones_mascaras.json")
print(f"  {len(sam_detections)} objetos | máscaras de {resultado.get('mask_shape', 'N/A')}")

# Para recuperar una máscara:
# H, W = resultado["mask_shape"]
# raw  = np.frombuffer(base64.b64decode(resultado["masks_b64"][0]), dtype=np.uint8)
# mask = np.unpackbits(raw)[:H*W].reshape(H, W).astype(bool)